In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
df = spark.read.format('delta').table("olympics.bronze.nocs")
df = df.withColumn("tag",F.split(F.column("tag"),"-")[0])
df.write.format("delta").mode("overwrite").option("path","abfss://olympics@adls7786.dfs.core.windows.net/silver/nocs").saveAsTable("olympics.silver.nocs")

In [0]:
df = spark.read.format("delta").table("olympics.bronze.athletes")
df = df.fillna({"birth_place":"xyz","birth_country":"xyz","residence_place":"xyz","residence_country":"aaa"})
df_sorted = df.sort("height","weight",ascending=[1,0]).where(F.col("weight") > 0)
df_data = df_sorted.withColumn("nationality",F.regexp_replace(F.col("nationality"),"United States","US"))
df_final = df_sorted.withColumnRenamed("code","athlete_id")
df_final = df_final.withColumn("occupation",F.split(F.col("occupation"),",")[0])
df_final.write.format("delta").mode("overwrite").option("path","abfss://olympics@adls7786.dfs.core.windows.net/silver/athletes").saveAsTable("olympics.silver.athletes")

In [0]:
df_window = df_final.select('athlete_id',
 'current',
 'name',
 'name_short',
 'name_tv',
 'gender',
 'function',
 'country_code',
 'country',
 'country_long',
 'nationality',
 'nationality_long',
 'nationality_code',
 'height',
 'weight')

df_window.withColumn("cum_weight",F.sum(F.col("weight")).over(Window.partitionBy("nationality").orderBy("height").rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()
